# Euro Football Discipline Stats Data Scraping

In this notebook we will describe all data sources needed for the Euro Football Discipline Stats project and how data acquisition was performed.

As the aim of the project is to analyse the distribution of yellow and red cards issued to club teams across national and international competitions, the minimum information needed is:

- \# of yellow cards
- \# of red cards
- \# fouls committed
- \# fouls drawn
- \# of games/minutes played

These are the minimum information, other data that might be useful could be:

- \# of players involved
- \# of tackles (defensive challenges measuring if a team has a more aggressive attitude)


To check anomalies in disciplirary data, we will work on dataset extracted from the three of the Major European Country Leagues (England, Spain, Germany, France, Italy) and all the major European Tournaments for Clubs: UEFA Champions League (UCL), UEFA Europe League (UEL), UEFA Europe Conference Leauge (UECL).  

There was not a single source where to retrieve these data and we used two different sources: [football-data.co.uk](https://www.football-data.co.uk/) for domestic leagues, [fbref.com](https://fbref.com/) for European Tournaments. The data set will be made homogeneous when we will perform data exploration and data wrangling steps in a different notebook.

## League Data

- source: [football-data.co.uk](https://www.football-data.co.uk/)
- Leagues: English Premier League, Spanish la Liga, Germany Bundesliga, France Ligue 1, Italian Serie A
- Starting season: 2011/12
- Method: inspect .csv files and extracting data

### Data Dictionary - Columns scraped:
- `Date`: Match Date (dd/mm/yy)
- `HomeTeam`: Home Team
- `AwayTeam`: Away Team
- `FTHG` - Results - Full Time Home Team Goals
- `FTAG`- Results - Full Time Away Team Goals
- `FTR` - Results - Full Time Result (H=Home Win, D=Draw, A=Away Win)
- `HTHG` - Results - Half Time Home Team Goals
- `HTAG` - Results - Half Time Away Team Goals
- `HTR` - Results - Half Time Result (H=Home Win, D=Draw, A=Away Win)
- `HF` - Discipline - Home Team Fouls Committed
- `AF` - Discipline - Away Team Fouls Committed
- `HY` - Discipline - Home Team Yellow Cards
- `AY` - Discipline - Away Team Yellow Cards
- `HR` - Discipline - Home Team Red Cards
- `AR` - Discipline - Away Team Red Cards
- `HS` - Attack/Defence - Home Team Shots
- `AS` - Attack/Defence - Away Team Shots
- `HST` - Attack/Defence - Home Team Shots on Target
- `AST` - Attack/Defence - Away Team Shots on Target
- `HC` - Attack/Defence - Home Team Corners
- `AC` - Attack/Defence - Away Team Corners

The following function implement the web scraping

In [2]:
import requests
import pandas as pd
from io import StringIO
import time

_DEFAULT_LEAGUES = {"Serie_A": "I1", 
                    "Premier_League": "E0", 
                    "La_Liga": "SP1",
                    "Bundesliga": "D1",
                    "Ligue_1": "F1",}

_DEFAULT_SEASONS = [
    "2526", "2425", "2324", "2223", "2122", 
    "2021", "1920", "1819", "1718", "1617", 
    "1516", "1415", "1314", "1213", "1112", 
]

_DEFAULT_COLS = [
    "Date", "HomeTeam", "AwayTeam",
    # Results
    "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR",
    # Discipline
    "HF", "AF", "HY", "AY", "HR", "AR",
    # Attack/Defence (optional)
    "HS", "AS", "HST", "AST", "HC", "AC"
]


def scrape_football_data(
    seasons: list = _DEFAULT_SEASONS,
    cols: list = _DEFAULT_COLS,
    leagues: dict = _DEFAULT_LEAGUES,
) -> dict:
    """Scrape match data from football-data.co.uk.

    Args:
        seasons: List of season codes, e.g. ["2324", "2223"].
        cols: List of columns to retain from each CSV.
        leagues: Dict mapping league name to football-data.co.uk code,
                 e.g. {"Serie_A": "I1"}.

    Returns:
        Dict mapping league name to a concatenated DataFrame.
    """

    def _fetch_league(code: str, season: str) -> pd.DataFrame | None:
        url = f"https://www.football-data.co.uk/mmz4281/{season}/{code}.csv"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            df = pd.read_csv(StringIO(r.text), on_bad_lines="skip")
            df["season"] = season
            return df
        print(f"❌ {url}")
        return None

    all_data = {}
    for name, code in leagues.items():
        frames = []
        for season in seasons:
            df = _fetch_league(code, season)
            if df is not None:
                available = [c for c in cols if c in df.columns]
                frames.append(df[available + ["season"]])
                print(f"✅ {name} {season}")
            time.sleep(0.5)  # be polite
        if frames:
            all_data[name] = pd.concat(frames, ignore_index=True)

    return all_data


# Usage - uncomment to run the function
# all_data = scrape_football_data()
# print(all_data["Serie_A"].head())

Saving data in `data/raw_data folder`

In [3]:
# uncomment if needed

# import pickle
# from pathlib import Path

# # Define path relative to notebook location
# output_path = Path("../data/raw/european_leagues_data.pkl")

# # Save
# with open(output_path, "wb") as f:
#     pickle.dump(all_data, f)

# print(f"✅ Saved to {output_path}")

Loading raw data from `data/raw_data` folder

In [4]:
import pickle
from pathlib import Path

with open("../data/raw/european_leagues_data.pkl", "rb") as f:
    all_data = pickle.load(f)

In [11]:
for key in all_data.keys():
    print(f"key name: {key}, dataframe shape: {all_data[key].shape}")

key name: Serie_A, dataframe shape: (5625, 22)
key name: Premier_League, dataframe shape: (5630, 22)
key name: La_Liga, dataframe shape: (5610, 22)
key name: Bundesliga, dataframe shape: (4527, 22)
key name: Ligue_1, dataframe shape: (5315, 22)


- Sanity Check

In [12]:
df_serie_A = all_data['Serie_A']

In [13]:
df_serie_A.dtypes

Date            str
HomeTeam        str
AwayTeam        str
FTHG        float64
FTAG        float64
FTR             str
HTHG        float64
HTAG        float64
HTR             str
HF          float64
AF          float64
HY          float64
AY          float64
HR          float64
AR          float64
HS          float64
AS          float64
HST         float64
AST         float64
HC          float64
AC          float64
season          str
dtype: object

In [14]:
df_serie_A[df_serie_A['HTHG'].isna()]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HF,...,AY,HR,AR,HS,AS,HST,AST,HC,AC,season
3357,28/08/16,Sassuolo,Pescara,0.0,3.0,A,NaN,NaN,NaN,12.0,...,2.0,0.0,0.0,13.0,15.0,3.0,5.0,2.0,7.0,1617
4100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1516
4481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1415
4896,23/09/12,Cagliari,Roma,0.0,3.0,A,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1213
5242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1213
5243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1213
5244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1213


In [15]:
df_bundesliga = all_data['Bundesliga']

In [16]:
df_bundesliga.isna().sum()

Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        1
HTAG        1
HTR         1
HF          1
AF          1
HY          1
AY          1
HR          1
AR          1
HS          1
AS          1
HST         1
AST         1
HC          1
AC          1
season      0
dtype: int64

In [17]:
df_bundesliga[df_bundesliga['HTHG'].isna()]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HF,...,AY,HR,AR,HS,AS,HST,AST,HC,AC,season
364,14/12/2024,Union Berlin,Bochum,0,2,A,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2425


In [18]:
df_Ligue_1 = all_data['Ligue_1']

In [19]:
df_Ligue_1.isna().sum()

Date        2
HomeTeam    2
AwayTeam    2
FTHG        2
FTAG        2
FTR         2
HTHG        3
HTAG        3
HTR         3
HF          5
AF          5
HY          3
AY          3
HR          3
AR          3
HS          3
AS          3
HST         3
AST         3
HC          3
AC          3
season      0
dtype: int64

In [20]:
df_Ligue_1[df_Ligue_1['HF'].isna()]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HF,...,AY,HR,AR,HS,AS,HST,AST,HC,AC,season
3358,16/04/17,Bastia,Lyon,0.0,3.0,A,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1617
3793,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1516
4554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1314
4993,18/09/11,Lyon,Marseille,2.0,0.0,H,2.0,0.0,H,NaN,...,3.0,0.0,0.0,10.0,6.0,5.0,3.0,3.0,5.0,1112
5106,17/12/11,Caen,Nancy,1.0,2.0,A,0.0,1.0,A,NaN,...,5.0,0.0,0.0,9.0,9.0,1.0,3.0,8.0,1.0,1112


In [21]:
df_Premier_League = all_data['Premier_League']

In [22]:
df_La_Liga = all_data['La_Liga']

In [23]:
df_Premier_League.isna().sum()

Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
HTHG        1
HTAG        1
HTR         1
HF          1
AF          1
HY          1
AY          1
HR          1
AR          1
HS          1
AS          1
HST         1
AST         1
HC          1
AC          1
season      0
dtype: int64

In [24]:
df_Premier_League[df_Premier_League['HR'].isna()]

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HF,...,AY,HR,AR,HS,AS,HST,AST,HC,AC,season
4489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1415


## European Competitions data

- source: [fbref.com](https://fbref.com/)
- European Competitions: UEFA Champions League (UCL), Europe League (UEL), Europe Conference League (UECL)
- Starting season: 2011/12 for UCL and UEL, 2021/22 for UECL
- Method: 
    1. Open each FBref URL manually in your browser - pass Cloudflare once
    2. Save page as HTML (Ctrl+S)
    3. Parse all HTML files locally with a simple Python script

The manual method was used after failing passing through Cloudflare authentication. The downloaded `.html` files are stored in `/data/raw/fbref_html` folder.

The produced script is `/src/scraper_local_html.py`. The produces `dict` is stored in `/data/raw/european_comps_data.pkl`

The produced dict contains three dataframes, one for each European competition.

Each dataframe contains 28 columns, each rows representing the discipline performances for a team partecipating the competition for that season.

### Data Dictionary - Columns scraped:

- `squad` - Team name
- `country_code` - Team country code  
- `num_pl` - number of players  
- `90s` - number of 90 minutes matches 
- `fls_committed` - number of fauls committed  
- `fld_drawn` - number of fauls received that lead to a shoot  
- `ycards_received` - number of yellow cards received  
- `rcards_received` - number of red cards received 
- `ycards_2nd_received` - number of second yellow cards received 
- `offsides_committed` - number of times the Team was on offside 
- `crosses_performed` - number of crosses performed 
- `interceptions_performed` - number of interceptions performed
- `tackles_won` - number of tackes won  
- `penalties_won` - number of penalties won  
- `og_scored` - number of own goal scored  
- `fls_won` - number of fauls drawn  
- `fld_conceded` - number of fauls committed that lead to a shoot 
- `ycards_caused` - number of yellow cards caused  
- `rcards_caused` - number of red cards caused 
- `ycards_2nd_caused` - number of second yellow cards caused
- `offsides_caused` - number of times the team succeed to offside the opponents  
- `crosses_faced` - number of crossed received 
- `interceptions_conceded` - number of interception conceded 
- `tackles_conceded` - number of tackles conceded 
- `penalties_conceded` - number of penalties conceded 
- `og_forced` - number of own goal forced  
- `competition` - competition name 
- `season` - season

> review `fld` part. 


In [25]:
with open("../data/raw/european_comps_data.pkl", "rb") as f:
    european_comps_data = pickle.load(f)

- Sanity Check

In [26]:
european_comps_data.keys()

dict_keys(['UCL', 'UEL', 'UECL'])

In [27]:
UCL_data = european_comps_data['UCL']

In [28]:
cols_visualized = ['squad', '90s', 'fls_committed', 'fls_won', 'fld_drawn', 'fld_conceded', 'season']

UCL_data[cols_visualized].head()

,squad,90s,fls_committed,fls_won,fld_drawn,fld_conceded,season
0,Arsenal,8.0,97,78,76,94,2016-2017
1,Atlético Madrid,12.0,124,124,122,117,2016-2017
2,Barcelona,10.0,123,145,137,119,2016-2017
3,Basel,6.0,109,67,68,105,2016-2017
4,Bayern Munich,10.3,120,96,90,102,2016-2017


In [30]:
from tabulate import tabulate

print(tabulate(UCL_data[UCL_data['season'] == '2016-2017'].head(), headers='keys', tablefmt='psql'))

+----+-----------------+----------------+----------+-------+-----------------+-------------+-------------------+-------------------+-----------------------+----------------------+---------------------+---------------------------+---------------+-----------------+-------------+-----------+----------------+-----------------+-----------------+---------------------+-------------------+-----------------+--------------------------+--------------------+----------------------+-------------+---------------+-----------+
|    | squad           | country_code   |   num_pl |   90s |   fls_committed |   fld_drawn |   ycards_received |   rcards_received |   ycards_2nd_received |   offsides_committed |   crosses_performed |   interceptions_performed |   tackles_won |   penalties_won |   og_scored |   fls_won |   fld_conceded |   ycards_caused |   rcards_caused |   ycards_2nd_caused |   offsides_caused |   crosses_faced |   interceptions_conceded |   tackles_conceded |   penalties_conceded |   og_force

## Other resources

- [the stats don't lie](https://www.thestatsdontlie.com/football/europe/italy/serie-a/fouls/)
- [fbref test](https://fbref.com/en/comps/8/2011-2012/misc/2011-2012-Champions-League-Stats)

In [32]:
30000000*0.04

1200000.0